In [1]:
import sys
# !{sys.executable} -m pip install datasets snac torchaudio soundfile huggingface_hub
# !{sys.executable} -m pip install torchaudio --index-url https://download.pytorch.org/whl/cu121
# !{sys.executable} -m pip install torchcodec
# !apt-get install -y libavutil-dev libavcodec-dev libavformat-dev ffmpeg
# !{sys.executable} -m pip install scipy
# !{sys.executable} -m pip install "datasets<3.0" soundfile librosa -q
# !{sys.executable} -m pip install transformers  scipy librosa -q

In [2]:
import os
import torch
print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA disponible: True
GPU: NVIDIA RTX 6000 Ada Generation


In [ ]:
from huggingface_hub import login
login("API_KEY")

In [4]:
from snac import SNAC
from datasets import load_dataset
from scipy.signal import resample_poly
from math import gcd

DATASET_NAME = "ylacombe/google-colombian-spanish"
DATASET_CONFIG = "female"
OUTPUT_DIR = "./colombian_tokenised"

ds = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")
ds_sample_rate = ds[0]["audio"]["sampling_rate"]

print(f"Sample rate: {ds_sample_rate}")
print(f"Columnas: {ds.column_names}")
print(f"Total ejemplos: {len(ds)}")

Sample rate: 48000
Columnas: ['audio', 'text', 'speaker_id']
Total ejemplos: 2369


In [5]:
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to("cuda")

/usr/local/lib/python3.11/dist-packages/snac/snac.py:108: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location="cpu")


In [6]:
def tokenise_audio(waveform):

    orig_freq = ds_sample_rate
    new_freq = 24000
    divisor = gcd(orig_freq, new_freq)
    up = new_freq // divisor
    down = orig_freq // divisor
    waveform_resampled = resample_poly(waveform, up, down).astype("float32")
    waveform_tensor = torch.from_numpy(waveform_resampled).unsqueeze(0).unsqueeze(0).to("cuda")

    with torch.inference_mode():
        codes = snac_model.encode(waveform_tensor)

    all_codes = []
    for i in range(codes[0].shape[1]):
        all_codes.append(codes[0][0][i].item() + 128266)
        all_codes.append(codes[1][0][2*i].item() + 128266 + 4096)
        all_codes.append(codes[2][0][4*i].item() + 128266 + (2*4096))
        all_codes.append(codes[2][0][(4*i)+1].item() + 128266 + (3*4096))
        all_codes.append(codes[1][0][(2*i)+1].item() + 128266 + (4*4096))
        all_codes.append(codes[2][0][(4*i)+2].item() + 128266 + (5*4096))
        all_codes.append(codes[2][0][(4*i)+3].item() + 128266 + (6*4096))

    return all_codes


def add_codes(example):
    codes_list = None
    try:
        answer_audio = example.get("audio")
        if answer_audio and "array" in answer_audio:
            codes_list = tokenise_audio(answer_audio["array"])
    except Exception as e:
        print(f"Skipping row: {e}")
        
    example["codes_list"] = codes_list
    return example


ds = ds.map(add_codes, remove_columns=["audio"])

Parameter 'function'=<function add_codes at 0x77fd30192160> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/2369 [00:00<?, ? examples/s]

In [7]:
ds = ds.filter(lambda x: x["codes_list"] is not None)
ds = ds.filter(lambda x: len(x["codes_list"]) > 0)
print(f"Ejemplos válidos: {len(ds)}")

Filter:   0%|          | 0/2369 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2369 [00:00<?, ? examples/s]

Ejemplos válidos: 2369


In [8]:
def remove_duplicate_frames(example):
    vals = example["codes_list"]
    if len(vals) % 7 != 0:
        vals = vals[:len(vals) - (len(vals) % 7)]

    result = vals[:7]
    for i in range(7, len(vals), 7):
        if vals[i] != result[-7]:
            result.extend(vals[i:i+7])

    example["codes_list"] = result
    return example

num_proc = max(1, os.cpu_count() - 2)
ds = ds.map(remove_duplicate_frames, num_proc=num_proc)

Map (num_proc=126):   0%|          | 0/2369 [00:00<?, ? examples/s]

In [9]:
from transformers import AutoTokenizer

tokenizer_name = "canopylabs/orpheus-3b-0.1-pretrained"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

tokeniser_length = 128256
start_of_text = 128000
end_of_text = 128009
start_of_speech = tokeniser_length + 1
end_of_speech = tokeniser_length + 2
start_of_human = tokeniser_length + 3
end_of_human = tokeniser_length + 4
start_of_ai = tokeniser_length + 5
end_of_ai = tokeniser_length + 6


def create_input_ids(example):
    
    tagged_text = f"colombia: {example['text']}"
    text_ids = tokenizer.encode(tagged_text, add_special_tokens=True)
    text_ids.append(end_of_text)
    n_text = len([start_of_human] + text_ids + [end_of_human] + [start_of_ai] + [start_of_speech])
    input_ids = (
        [start_of_human] + text_ids + [end_of_human] +
        [start_of_ai] + [start_of_speech] +
        example["codes_list"] + [end_of_speech] + [end_of_ai]
    )

    labels = [-100] * n_text + example["codes_list"] + [end_of_speech, end_of_ai]

    example["input_ids"]      = input_ids
    example["labels"]         = labels
    example["attention_mask"] = [1] * len(input_ids)

    return example


ds = ds.map(create_input_ids, num_proc=num_proc, remove_columns=["text", "codes_list"])

Map (num_proc=126):   0%|          | 0/2369 [00:00<?, ? examples/s]

In [10]:
columns_to_keep = ["input_ids", "labels", "attention_mask"]
columns_to_remove = [col for col in ds.column_names if col not in columns_to_keep]
ds = ds.remove_columns(columns_to_remove)

print(f"Columnas finales: {ds.column_names}")
print(f"Ejemplo longitud input_ids: {len(ds[0]['input_ids'])}")

Columnas finales: ['input_ids', 'labels', 'attention_mask']
Ejemplo longitud input_ids: 346


In [11]:
ds.save_to_disk(OUTPUT_DIR)
print(OUTPUT_DIR)
print(f"Total ejemplos: {len(ds)}")

Saving the dataset (0/1 shards):   0%|          | 0/2369 [00:00<?, ? examples/s]

./colombian_tokenised
Total ejemplos: 2369
